### **Semana 2 - Inferencia: logits, decoding, contexto y KV cache**

#### **Propósito**

Este cuaderno estudia el camino que sigue un Transformer causal después de producir una representación contextual.

```text
hidden state -> LM head -> logits -> softmax -> decoding -> nuevo token -> contexto ampliado -> autoregresión
```

Luego conecta generación con eficiencia:

```text
contexto creciente -> K/V históricos -> KV cache -> memoria y tráfico -> MHA /GQA/SWA/MLA
```

El cuaderno combina tres niveles:

1. mecanismo autocontenido con tensores pequeños,
2. extensión opcional con `distilgpt2`,
3. estimación analítica del KV cache.

La parte autocontenida puede ejecutarse sin descargar modelos.

#### **1. Configuración y reproducibilidad**

Usaremos PyTorch para los mecanismos mínimos y pandas/matplotlib para registrar evidencia.

Las funciones tienen nombres en inglés porque representan interfaces de código, comentarios y cadenas se mantienen en español.

In [ ]:
import math
import random
import time
from typing import Dict, Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

#### **2. De la representación contextual a los logits**

Sea $h_t\in\mathbb{R}^{d}$ la representación de la última posición.

Un head de lenguaje puede proyectarla al vocabulario:

$$
z_t = h_t W^\top + b,
$$

donde:

$$
z_t\in\mathbb{R}^{|\mathcal V|}.
$$

$z_t$ contiene **logits**.

Los logits:

- no son probabilidades,
- no tienen que sumar 1,
- pueden ser negativos,
- solo adquieren interpretación probabilística después de una normalización como `softmax`.

En algunas arquitecturas los pesos de salida están ligados a la matriz de embeddings. Esa decisión no es universal.

In [ ]:
toy_vocab = [
    "<bos>",
    "los",
    "modelos",
    "usan",
    "contexto",
    "memoria",
    "eficiente",
    ".",
]

hidden_state = torch.tensor(
    [[0.8, -0.4, 1.1, 0.2]],
    dtype=torch.float32,
)

torch.manual_seed(SEED)
lm_head = torch.randn(
    len(toy_vocab),
    hidden_state.size(-1),
) * 0.7
bias = torch.linspace(
    -0.2,
    0.2,
    steps=len(toy_vocab),
)

logits = hidden_state @ lm_head.T + bias

tabla_logits = pd.DataFrame({
    "token": toy_vocab,
    "logit": logits[0].tolist(),
}).sort_values("logit", ascending=False)

tabla_logits

#### **3. Logits -> softmax -> distribución**

La distribución del siguiente token es:

$$
p(v\mid x_{1:t})
=
\frac{\exp(z_v)}
{\sum_u \exp(z_u)}.
$$

La invariante mínima es:

$$
\sum_v p(v\mid x_{1:t}) \approx 1.
$$

`softmax` normaliza scores. No decide causalidad y no decide qué token será finalmente seleccionado.

In [ ]:
probs = torch.softmax(logits, dim=-1)

assert torch.allclose(
    probs.sum(dim=-1),
    torch.ones(1),
    atol=1e-6,
)

tabla_probs = pd.DataFrame({
    "token": toy_vocab,
    "logit": logits[0].tolist(),
    "probabilidad": probs[0].tolist(),
}).sort_values("probabilidad", ascending=False)

tabla_probs

#### **4. Temperatura**

La temperatura modifica la escala de los logits antes de `softmax`:

$$
p_\tau(v)
=
\frac{\exp(z_v/\tau)}
{\sum_u \exp(z_u/\tau)}.
$$

Interpretación:

```text
tau < 1
  -> distribución más concentrada

tau = 1
  -> distribución original

tau > 1
  -> distribución más plana
```

Una forma de medir concentración es la entropía:

$$
H(p)
=
-\sum_v p(v)\log_2 p(v).
$$

In [ ]:
def probabilities_from_logits(
    logits_1d: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    # Convierte logits 1D en probabilidades con temperatura.
    if temperature <= 0:
        raise ValueError("temperature debe ser > 0")
    return torch.softmax(
        logits_1d / temperature,
        dim=-1,
    )


def entropy_bits(
    probs_1d: torch.Tensor,
) -> float:
    # Calcula entropía en bits.
    p = probs_1d.clamp_min(1e-12)
    return float(
        -(p * torch.log2(p)).sum().item()
    )


filas = []
for temperature in [0.5, 0.8, 1.0, 1.5, 2.0]:
    p = probabilities_from_logits(
        logits[0],
        temperature,
    )
    filas.append({
        "temperature": temperature,
        "entropia_bits": entropy_bits(p),
        "p_max": float(p.max().item()),
        "token_top1": toy_vocab[
            int(p.argmax().item())
        ],
    })

pd.DataFrame(filas)

#### 5. **Top-k y top-p desde cero**

**Top-k** conserva un número fijo de candidatos.

**Top-p** conserva el conjunto mínimo cuya masa acumulada alcanza el umbral $p$.

```text
top-k -> cantidad fija de candidatos

top-p -> cantidad dinámica de candidatos
```

El filtrado se realiza sobre logits y después se vuelve a aplicar `softmax`.

In [ ]:
def apply_top_k(
    logits_1d: torch.Tensor,
    top_k: Optional[int] = None,
) -> torch.Tensor:
    # Conserva los top-k logits y enmascara el resto.
    if top_k is None or top_k <= 0:
        return logits_1d.clone()

    k = min(
        top_k,
        logits_1d.numel(),
    )
    values, _ = torch.topk(
        logits_1d,
        k=k,
    )
    threshold = values[-1]

    return torch.where(
        logits_1d < threshold,
        torch.full_like(
            logits_1d,
            float("-inf"),
        ),
        logits_1d,
    )


def apply_top_p(
    logits_1d: torch.Tensor,
    top_p: Optional[float] = None,
) -> torch.Tensor:
    # Conserva el núcleo mínimo de masa acumulada >= top_p.
    if top_p is None or top_p >= 1.0:
        return logits_1d.clone()
    if top_p <= 0:
        raise ValueError(
            "top_p debe estar en (0, 1]"
        )

    sorted_logits, sorted_indices = torch.sort(
        logits_1d,
        descending=True,
    )
    sorted_probs = torch.softmax(
        sorted_logits,
        dim=-1,
    )
    cumulative_probs = torch.cumsum(
        sorted_probs,
        dim=-1,
    )

    remove = cumulative_probs > top_p
    remove[1:] = remove[:-1].clone()
    remove[0] = False

    sorted_logits = sorted_logits.masked_fill(
        remove,
        float("-inf"),
    )

    filtered = torch.full_like(
        logits_1d,
        float("-inf"),
    )
    filtered.scatter_(
        0,
        sorted_indices,
        sorted_logits,
    )
    return filtered


def sample_next_token(
    logits_1d: torch.Tensor,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> int:
    # Muestrea un token después de temperatura y filtrado.
    scaled = logits_1d / max(
        temperature,
        1e-6,
    )
    filtered = apply_top_k(
        scaled,
        top_k,
    )
    filtered = apply_top_p(
        filtered,
        top_p,
    )
    probs = torch.softmax(
        filtered,
        dim=-1,
    )
    return int(
        torch.multinomial(
            probs,
            num_samples=1,
        ).item()
    )


def candidate_count(
    filtered_logits: torch.Tensor,
) -> int:
    # Cuenta candidatos no enmascarados.
    return int(
        torch.isfinite(
            filtered_logits
        ).sum().item()
    )

In [ ]:
demo_logits = logits[0]

comparacion = []

for nombre, temperature, top_k, top_p in [
    ("Sampling", 1.0, None, None),
    ("Temperature=0.7", 0.7, None, None),
    ("Top-k=3", 1.0, 3, None),
    ("Top-p=0.8", 1.0, None, 0.8),
]:
    scaled = demo_logits / temperature
    filtered = apply_top_k(
        scaled,
        top_k,
    )
    filtered = apply_top_p(
        filtered,
        top_p,
    )
    p = torch.softmax(
        filtered,
        dim=-1,
    )

    comparacion.append({
        "estrategia": nombre,
        "candidatos": candidate_count(
            filtered
        ),
        "entropia_bits": entropy_bits(p),
        "p_max": float(
            p.max().item()
        ),
    })

pd.DataFrame(comparacion)

#### **6. Autoregresión**

Un modelo causal aplica repetidamente:

$$
x_{1:t}
\mapsto
p(x_{t+1}\mid x_{1:t}).
$$

Luego una política elige $\hat{x}_{t+1}$ y el nuevo token pasa a formar parte del contexto.

La siguiente simulación usa una tabla de transiciones pequeña. No pretende modelar lenguaje real. Su única función es hacer visible el loop de inferencia.

In [ ]:
# Logits de transición didácticos.
# Cada fila representa el token actual y cada columna el siguiente token.
#
# Nota de diseño: la versión inicial producía distribuciones demasiado
# concentradas en varias transiciones. Con la seed usada en el ejemplo,
# temperature, top-k y top-p podían terminar generando la misma secuencia,
# aunque los mecanismos de decoding fueran distintos.
#
# Esta versión escala uniformemente los logits a la mitad. El escalado
# conserva el argmax de cada fila y, por tanto, mantiene el mismo recorrido
# bajo greedy decoding, pero reduce la concentración de la distribución.
# Esto hace más observable el efecto de distintas políticas de sampling
# en un ejemplo pequeño y reproducible.
#
# Matemáticamente, softmax(0.5 * z) es equivalente a aplicar temperatura
# T=2 sobre los logits originales. Aquí se usa solo para calibrar la
# distribución base de este modelo didáctico antes del experimento.
# No constituye una recomendación de temperatura para un modelo real.
transition_logits = torch.tensor([
    [-2.0,  2.0,  0.5, -1.0, -1.0, -1.0, -1.0, -1.5],
    [-2.0, -1.0,  2.0,  0.5,  0.0, -0.5, -0.5, -1.0],
    [-2.0, -1.0, -0.5,  2.0,  0.5,  0.25, -0.5, -1.0],
    [-2.0, -1.0, -1.0, -0.5,  1.5,  1.35,  0.25, -1.0],
    [-2.0, -1.0, -1.0, -1.0, -0.5,  1.25,  1.5,  0.5],
    [-2.0, -1.0, -1.0, -1.0,  0.5, -0.5,  1.6,  0.5],
    [-2.0, -1.0, -1.0, -1.0, -0.5, -0.5, -0.5,  2.0],
    [-2.0,  1.0,  0.75, -1.0, -1.0, -1.0, -1.0, -0.5],
], dtype=torch.float32)


def generate_toy(
    max_new_tokens: int = 10,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
) -> str:
    # Genera una secuencia token por token.
    token_id = 0
    generated = [token_id]

    for _ in range(max_new_tokens):
        next_logits = transition_logits[
            token_id
        ]
        token_id = sample_next_token(
            next_logits,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )
        generated.append(token_id)

    tokens = [
        toy_vocab[i]
        for i in generated
    ]
    return " ".join(tokens)


for cfg in [
    {"temperature": 0.7},
    {"temperature": 1.0, "top_k": 2},
    {"temperature": 1.0, "top_p": 0.85},
]:
    torch.manual_seed(SEED)
    print(
        cfg,
        "->",
        generate_toy(**cfg),
    )

#### **7. Modelo causal preentrenado opcional**

La mecánica anterior no depende de Hugging Face.

Para conectar con un modelo real se puede usar `distilgpt2`.

**Importante:**

- GPT-2/DistilGPT2 no es un modelo instruct moderno,
- se usa para inspeccionar inferencia, no para juzgar capacidad actual de LLM,
- por defecto el cuaderno solo usa archivos locales de cache,
- cambia `ALLOW_DOWNLOAD=True` si deseas permitir la descarga.

Esto permite que `make execute-cuaderno2` siga siendo robusto en un entorno sin red.

In [ ]:
ALLOW_DOWNLOAD = False
MODEL_NAME = "distilgpt2"

HF_READY = False
tokenizer = None
model = None

try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME,
            local_files_only=not ALLOW_DOWNLOAD,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            local_files_only=not ALLOW_DOWNLOAD,
        ).to(DEVICE)
        model.eval()

        if tokenizer.pad_token is None:
            tokenizer.pad_token = (
                tokenizer.eos_token
            )

        HF_READY = True
        print("Modelo listo:", MODEL_NAME)
    except Exception as exc:
        print(
            "Modelo Hugging Face no disponible "
            "en cache local."
        )
        print(
            "Para descargarlo, cambia "
            "ALLOW_DOWNLOAD=True y vuelve a ejecutar."
        )
        print(
            "Detalle:",
            type(exc).__name__,
        )
except Exception as exc:
    print("transformers no está instalado.")
    print(
        "Instala las dependencias de Semana 2."
    )
    print(
        "Detalle:",
        type(exc).__name__,
    )

print("HF_READY =", HF_READY)

In [ ]:
def top_next_tokens_hf(
    prompt: str,
    k: int = 10,
) -> pd.DataFrame:
    # Inspecciona la distribución local del siguiente token.
    if not HF_READY:
        return pd.DataFrame({
            "estado": [
                "Modelo no disponible, "
                "sección opcional omitida."
            ]
        })

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        next_logits = (
            outputs.logits[:, -1, :]
        )
        probs = torch.softmax(
            next_logits,
            dim=-1,
        )
        values, indices = torch.topk(
            probs,
            k=k,
            dim=-1,
        )

    ids = indices[0].tolist()

    return pd.DataFrame({
        "token": tokenizer.convert_ids_to_tokens(
            ids
        ),
        "texto": [
            tokenizer.decode([i])
            for i in ids
        ],
        "probabilidad": (
            values[0]
            .detach()
            .cpu()
            .tolist()
        ),
    })


top_next_tokens_hf(
    "Artificial intelligence systems can",
    k=10,
)

#### **8. Decoding con un modelo real**

La función siguiente mantiene fijo el modelo y cambia únicamente la política.

Esto permite separar:

```text
modelo !=decoding
```

In [ ]:
def generate_hf(
    prompt: str,
    max_new_tokens: int = 30,
    do_sample: bool = True,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
    use_cache: bool = True,
) -> str:
    # Genera texto con una configuración explícita.
    if not HF_READY:
        return (
            "[Modelo Hugging Face "
            "no disponible]"
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(DEVICE)

    kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "use_cache": use_cache,
        "pad_token_id": (
            tokenizer.eos_token_id
        ),
    }

    if do_sample:
        kwargs["temperature"] = temperature
        if top_k > 0:
            kwargs["top_k"] = top_k
        if top_p < 1.0:
            kwargs["top_p"] = top_p

    with torch.no_grad():
        output = model.generate(
            **inputs,
            **kwargs,
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )


if HF_READY:
    prompt_real = (
        "Artificial intelligence systems can"
    )

    configs = [
        {
            "nombre": "Greedy",
            "do_sample": False,
        },
        {
            "nombre": "Sampling",
            "do_sample": True,
            "temperature": 1.0,
        },
        {
            "nombre": "Temperature=0.7",
            "do_sample": True,
            "temperature": 0.7,
        },
        {
            "nombre": "Top-k=40",
            "do_sample": True,
            "top_k": 40,
        },
        {
            "nombre": "Top-p=0.9",
            "do_sample": True,
            "top_p": 0.9,
        },
    ]

    filas = []
    for cfg in configs:
        cfg_call = {
            k: v
            for k, v in cfg.items()
            if k != "nombre"
        }
        torch.manual_seed(SEED)
        filas.append({
            "estrategia": cfg["nombre"],
            "salida": generate_hf(
                prompt_real,
                **cfg_call,
            ),
        })

    display(pd.DataFrame(filas))
else:
    print("Sección omitida: HF_READY=False")

#### **9. Ventana de contexto**

La longitud relevante es la longitud tokenizada.

Una restricción simple es:

$$
T_{\mathrm{prompt}}
+
T_{\mathrm{historial}}
+
T_{\mathrm{salida}}
\le
L_{\max}.
$$

La ventana de contexto es un **presupuesto**.

Aumentar contexto no es gratis:

```text
más tokens -> más cómputo -> más estado K/V -> más memoria
```

Semana 3 estudiará cómo decidir qué información merece ocupar ese presupuesto.

In [ ]:
def count_tokens_hf(
    text: str,
) -> Optional[int]:
    # Cuenta tokens reales si el tokenizer está disponible.
    if not HF_READY:
        return None
    return len(
        tokenizer(
            text,
            add_special_tokens=False,
        )["input_ids"]
    )


ejemplos = [
    "Explain KV cache.",
    (
        "Explain KV cache and compare it "
        "with recomputing the full prefix."
    ),
    (
        "Explain KV cache, define what "
        "keys and values are stored, and "
        "state one limitation of using a "
        "larger context."
    ),
]

for text in ejemplos:
    n = count_tokens_hf(text)
    print({
        "texto": text,
        "tokens": (
            n
            if n is not None
            else "requiere tokenizer"
        ),
    })

#### **10. KV cache: qué se reutiliza**

En cada capa causal, para los tokens previos ya se calcularon:

$$
K_{1:t},
\qquad
V_{1:t}.
$$

Al generar el token $t+1$, el sistema puede conservar esos tensores y calcular únicamente el estado nuevo.

Sin cache, una implementación ingenua vuelve a procesar repetidamente el prefijo.

Con cache:

```text
tokens previos
  -> K/V reutilizados

token nuevo
  -> Q nuevo
  -> K nuevo
  -> V nuevo
```

El cache intercambia memoria por reducción de cómputo redundante.

#### **11. Memoria lógica del KV cache**

Una aproximación didáctica para MHA/GQA/MQA es:

$$
M_{\mathrm{KV}}
=
B L T \, 2 H_{\mathrm{KV}} d_h b.
$$

Donde:

- $B$: batch,
- $L$: número de capas,
- $T$: tokens conservados,
- $H_{\mathrm{KV}}$: número de KV heads,
- $d_h$: dimensión por head,
- $b$: bytes por elemento,
- `2`: keys y values.

Esta fórmula estima el estado K/V. No representa la memoria total de inferencia.

In [ ]:
def estimate_kv_cache_bytes(
    batch_size: int,
    num_layers: int,
    context_length: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_value: int = 2,
) -> int:
    # Estima bytes lógicos de K y V.
    return (
        batch_size
        * num_layers
        * context_length
        * 2
        * num_kv_heads
        * head_dim
        * bytes_per_value
    )


def bytes_to_gb(
    value: int,
) -> float:
    # Convierte bytes a GB decimal.
    return value / 1_000_000_000


def bytes_to_gib(
    value: int,
) -> float:
    # Convierte bytes a GiB.
    return value / (1024 ** 3)


config = {
    "batch_size": 1,
    "num_layers": 32,
    "query_heads": 32,
    "head_dim": 128,
    "bytes_per_value": 2,
}

filas = []

for context_length in [
    4096,
    8192,
    32768,
    131072,
]:
    for variante, kv_heads in [
        ("MHA", 32),
        ("GQA", 8),
        ("MQA", 1),
    ]:
        value = estimate_kv_cache_bytes(
            batch_size=config["batch_size"],
            num_layers=config["num_layers"],
            context_length=context_length,
            num_kv_heads=kv_heads,
            head_dim=config["head_dim"],
            bytes_per_value=(
                config["bytes_per_value"]
            ),
        )
        filas.append({
            "contexto": context_length,
            "variante": variante,
            "kv_heads": kv_heads,
            "GB": bytes_to_gb(value),
            "GiB": bytes_to_gib(value),
        })

tabla_cache = pd.DataFrame(filas)
tabla_cache

In [ ]:
for variante in [
    "MHA",
    "GQA",
    "MQA",
]:
    subset = tabla_cache[
        tabla_cache["variante"] == variante
    ]
    plt.plot(
        subset["contexto"],
        subset["GiB"],
        marker="o",
        label=variante,
    )

plt.xlabel("Longitud de contexto")
plt.ylabel("KV cache estimado (GiB)")
plt.title("Crecimiento lógico del KV cache")
plt.legend()
plt.show()

#### **12. GQA como reducción del número de KV heads**

En MHA:

```text
query_heads = kv_heads
```

En GQA:

```text
query_heads > kv_heads > 1
```

En MQA:

```text
kv_heads = 1
```

Bajo la fórmula anterior y manteniendo todo lo demás fijo:

$$
\frac{M_{\mathrm{GQA}}}
{M_{\mathrm{MHA}}}
=
\frac{H_{\mathrm{KV}}}
{H_Q}.
$$

Con 32 query heads y 8 KV heads:

$$
\frac{8}{32}=0.25.
$$

La reducción corresponde al componente K/V modelado, no a toda la memoria del proceso.

In [ ]:
context = 131072

mha = estimate_kv_cache_bytes(
    1,
    32,
    context,
    32,
    128,
    2,
)

gqa = estimate_kv_cache_bytes(
    1,
    32,
    context,
    8,
    128,
    2,
)

print(
    "MHA GiB:",
    round(bytes_to_gib(mha), 3),
)
print(
    "GQA GiB:",
    round(bytes_to_gib(gqa), 3),
)
print(
    "Razón GQA/MHA:",
    round(gqa / mha, 3),
)

assert math.isclose(
    gqa / mha,
    8 / 32,
    rel_tol=1e-12,
)

#### **13. SWA y MLA: modelos distintos de reducción**

**Sliding Window Attention (SWA)** limita cuántas posiciones históricas son visibles en una capa local.

Bajo el supuesto de que la implementación conserva únicamente la ventana activa:

$$
T_{\mathrm{efectivo}}
=
\min(T,W).
$$

**MLA** comprime el estado que representa K/V.

Attention AI Lab usa una aproximación conceptual:

$$
M_{\mathrm{MLA}}
=
B L T \, 2 r b,
$$

donde $r$ es el rango latente.

**Nota de consistencia entre materiales:** el notebook de MCC225 usa una toy MLA que guarda un único vector latente por token/capa y, por tanto, cuenta el cache de forma diferente. Los valores numéricos de ambos modelos didácticos no deben mezclarse como si fueran la misma implementación.

In [ ]:
def estimate_swa_cache_bytes(
    batch_size: int,
    num_layers: int,
    context_length: int,
    window_size: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_value: int = 2,
) -> int:
    # Estimación SWA bajo cache truncado a la ventana activa.
    effective_context = min(
        context_length,
        window_size,
    )
    return estimate_kv_cache_bytes(
        batch_size=batch_size,
        num_layers=num_layers,
        context_length=effective_context,
        num_kv_heads=num_kv_heads,
        head_dim=head_dim,
        bytes_per_value=bytes_per_value,
    )


def estimate_mla_cache_bytes(
    batch_size: int,
    num_layers: int,
    context_length: int,
    latent_rank: int,
    bytes_per_value: int = 2,
) -> int:
    # Modelo conceptual de Attention AI Lab: 2 * rango latente.
    return (
        batch_size
        * num_layers
        * context_length
        * 2
        * latent_rank
        * bytes_per_value
    )


context = 131072

escenarios = {
    "MHA": estimate_kv_cache_bytes(
        1, 32, context, 32, 128, 2
    ),
    "GQA": estimate_kv_cache_bytes(
        1, 32, context, 8, 128, 2
    ),
    "SWA": estimate_swa_cache_bytes(
        1, 32, context, 4096, 32, 128, 2
    ),
    "MLA conceptual": estimate_mla_cache_bytes(
        1, 32, context, 512, 2
    ),
}

pd.DataFrame([
    {
        "variante": nombre,
        "GB": bytes_to_gb(value),
        "GiB": bytes_to_gib(value),
    }
    for nombre, value in escenarios.items()
])

#### 14. **`use_cache=True` frente a `use_cache=False`**

Si el modelo real está disponible, se puede medir una diferencia empírica de tiempo.

La medición:

- hace calentamiento,
- sincroniza CUDA cuando corresponde,
- repite varias veces,
- no debe interpretarse como benchmark productivo.

Una sola medición en una laptop o GPU no demuestra comportamiento universal.

In [ ]:
def sync_device() -> None:
    # Sincroniza CUDA antes de medir, si corresponde.
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def timed_generation(
    prompt: str,
    use_cache: bool,
    reps: int = 2,
    max_new_tokens: int = 20,
) -> Dict[str, float]:
    # Mide tiempos simples de generación greedy.
    if not HF_READY:
        return {
            "use_cache": use_cache,
            "promedio_s": float("nan"),
            "min_s": float("nan"),
            "max_s": float("nan"),
        }

    _ = generate_hf(
        prompt,
        max_new_tokens=5,
        do_sample=False,
        use_cache=use_cache,
    )
    sync_device()

    times = []

    for _ in range(reps):
        sync_device()
        start = time.perf_counter()

        _ = generate_hf(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=use_cache,
        )

        sync_device()
        times.append(
            time.perf_counter() - start
        )

    return {
        "use_cache": use_cache,
        "promedio_s": float(
            np.mean(times)
        ),
        "min_s": float(
            np.min(times)
        ),
        "max_s": float(
            np.max(times)
        ),
    }


if HF_READY:
    prompt_cache = (
        "KV cache improves autoregressive "
        "inference because"
    )

    display(pd.DataFrame([
        timed_generation(
            prompt_cache,
            True,
        ),
        timed_generation(
            prompt_cache,
            False,
        ),
    ]))
else:
    print(
        "Benchmark omitido: HF_READY=False"
    )

#### **15. Attention AI Lab como sistema de inspección**

El cuaderno produce las fórmulas.

Attention AI Lab permite llevarlas a un sistema interactivo:

```text
fórmula
  ->
escenario
  ->
estimador
  ->
JSON / Markdown
  ->
discusión
```

Para Semana 2 interesan:

- KV Cache Estimator,
- 32k, 64k, 128k y 1M tokens,
- MHA,
- GQA,
- SWA,
- MLA,
- learning path `Entiende KV Cache en 12 minutos`.

Repositorio:

https://github.com/kapumota/attentionlab-ai

#### 16. **Qué no debe confundirse**

```text
temperature
  -> cambia distribución de sampling

top-k/top-p
  -> cambian conjunto candidato

KV cache
  -> reutiliza estado

GQA
  -> reduce KV heads

SWA
  -> reduce rango atendido en capas locales

MLA
  -> comprime representación del cache

FlashAttention
  -> reorganiza IO del cálculo exacto de attention
```

Estas técnicas pueden coexistir porque actúan en niveles diferentes.

#### **17. Preguntas de refuerzo

1. ¿Cuál es la diferencia entre logit y probabilidad?
2. ¿Qué invariante debe cumplir la salida de `softmax`?
3. ¿Por qué temperatura no modifica los pesos del modelo?
4. ¿Qué diferencia top-k de top-p?
5. ¿Por qué top-p puede usar distinto número de candidatos en cada paso?
6. ¿Qué significa generación autoregresiva?
7. ¿Por qué una mala selección local puede afectar pasos posteriores?
8. ¿Qué ocupa la ventana de contexto?
9. ¿Qué se guarda exactamente en KV cache?
10. ¿Por qué Q histórico no se cachea de la misma forma que K/V?
11. ¿Por qué GQA reduce el componente K/V?
12. ¿Qué supuesto adicional se usa al estimar SWA?
13. ¿Qué diferencia una estimación de memoria de un benchmark físico?
14. ¿FlashAttention reduce necesariamente el tamaño lógico del KV cache?.

#### **Cierre**

La Semana 2 puede resumirse como:

```text
representación -> logits -> distribución -> decoding -> token -> estado reutilizable
  ->
costo de inferencia
```

La pregunta ya no es solo:

> ¿Qué calcula el modelo?.

También es:

> ¿Cómo se usa esa distribución y cuánto cuesta mantener el estado necesario para seguir generando?.